# Lab 05 — Spark Structured Streaming
**Lifecycle, Kafka, event time, watermark, checkpoint, dan state**

Notebook ini mengikuti Deck 05 slides 37–40. Semua query harus dihentikan dengan `stop()` sebelum berpindah lab.

## Capaian praktik
- Menjelaskan lifecycle micro-batch dan membaca progress query.
- Membaca Kafka, mem-parsing JSON, dan mempertahankan metadata offset.
- Membuktikan checkpoint resume dan mengamati late data terhadap watermark/state.
- Merancang pipeline Kafka → validasi → window → sink.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from datetime import datetime, timedelta, timezone
import json, os, shutil, time

spark = (
    SparkSession.builder
    .appName("spark-training-lab-05")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

## Helper — hentikan query aktif

In [ ]:
for query in spark.streams.active:
    print("Stopping:", query.name, query.id)
    query.stop()
print("Active queries:", len(spark.streams.active))

## Lab 1 — Streaming tanpa Kafka: lifecycle + progress (15 menit)
**Rujukan slide:** Deck 05, slide 37.

Buat rate source 20 rows/s, kelompokkan `value % 3`, lalu amati memory sink dan `lastProgress`.

In [ ]:
rate_df = spark.readStream.format("rate").option("rowsPerSecond", 20).load()
counts = rate_df.withColumn("bucket", F.col("value") % 3).groupBy("bucket").count()
print("isStreaming sebelum start:", counts.isStreaming)

In [ ]:
q1 = (
    counts.writeStream.format("memory")
    .queryName("lab_rate_counts")
    .outputMode("complete")
    .trigger(processingTime="2 seconds")
    .start()
)
time.sleep(8)
spark.sql("SELECT * FROM lab_rate_counts ORDER BY bucket").show()
progress = q1.lastProgress or {}
print(json.dumps({k: progress.get(k) for k in ["inputRowsPerSecond", "processedRowsPerSecond", "durationMs"]}, indent=2))
q1.stop()

## Lab 2 — Kafka → Spark → parsed output + checkpoint (25 menit)
**Rujukan slide:** Deck 05, slide 38.

Topic `transactions` disiapkan oleh Compose. Connector sudah dibundel pada driver dan executor; tidak ada download Maven saat kelas.

In [ ]:
kafka_bootstrap = "kafka:9092"
kafka_schema = T.StructType([
    T.StructField("trx_id", T.StringType()),
    T.StructField("region", T.StringType()),
    T.StructField("amount", T.DoubleType()),
    T.StructField("event_time", T.TimestampType()),
])

def send_events(rows):
    payload = spark.createDataFrame(rows, ["trx_id", "region", "amount", "event_time"])
    (payload.selectExpr(
        "CAST(trx_id AS STRING) AS key",
        "to_json(named_struct('trx_id', trx_id, 'region', region, 'amount', amount, 'event_time', event_time)) AS value"
    ).write.format("kafka")
     .option("kafka.bootstrap.servers", kafka_bootstrap)
     .option("topic", "transactions")
     .save())

raw_kafka = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load()
)
parsed = (
    raw_kafka
    .selectExpr("CAST(value AS STRING) AS json", "topic", "partition", "offset", "timestamp AS kafka_timestamp")
    .withColumn("data", F.from_json("json", kafka_schema))
    .select("data.*", "topic", "partition", "offset", "kafka_timestamp")
)
parsed.printSchema()

In [ ]:
checkpoint_local = "/home/spark/outputs/streaming_lab/kafka_checkpoint"
if os.path.exists(checkpoint_local):
    shutil.rmtree(checkpoint_local)

q2 = (
    parsed.writeStream.format("console")
    .queryName("lab_kafka_parsed")
    .outputMode("append")
    .option("truncate", False)
    .option("checkpointLocation", f"file://{checkpoint_local}")
    .trigger(processingTime="2 seconds")
    .start()
)
now = datetime.now(timezone.utc)
send_events([
    ("K001", "JKT", 100000.0, now),
    ("K002", "BDG", 250000.0, now + timedelta(seconds=1)),
])
time.sleep(8)
print(json.dumps(q2.lastProgress or {}, indent=2)[:6000])
q2.stop()

In [ ]:
q2_restart = (
    parsed.writeStream.format("console")
    .queryName("lab_kafka_parsed")
    .outputMode("append")
    .option("truncate", False)
    .option("checkpointLocation", f"file://{checkpoint_local}")
    .trigger(processingTime="2 seconds")
    .start()
)
send_events([("K003", "SBY", 300000.0, datetime.now(timezone.utc))])
time.sleep(8)
print(json.dumps(q2_restart.lastProgress or {}, indent=2)[:6000])
q2_restart.stop()

## Lab 3 — Window + watermark + controlled late data (30 menit)
**Rujukan slide:** Deck 05, slide 39.

Gunakan window satu menit dan watermark dua menit. Event berwaktu maju digunakan untuk menggerakkan watermark sebelum event terlambat dikirim.

In [ ]:
watermark_checkpoint_local = "/home/spark/outputs/streaming_lab/watermark_checkpoint"
if os.path.exists(watermark_checkpoint_local):
    shutil.rmtree(watermark_checkpoint_local)

late_agg = (
    parsed.filter(F.col("trx_id").startswith("W") & F.col("event_time").isNotNull())
    .withWatermark("event_time", "2 minutes")
    .groupBy(F.window("event_time", "1 minute"), "region")
    .agg(F.count("*").alias("n"), F.sum("amount").alias("total_amount"))
)
q3 = (
    late_agg.writeStream.format("memory")
    .queryName("lab_watermark")
    .outputMode("update")
    .option("checkpointLocation", f"file://{watermark_checkpoint_local}")
    .trigger(processingTime="2 seconds")
    .start()
)

base_time = datetime.now(timezone.utc)
send_events([("W001", "JKT", 100000.0, base_time)])
time.sleep(5)
send_events([("W002", "JKT", 200000.0, base_time + timedelta(minutes=5))])
time.sleep(5)
send_events([("W-LATE", "JKT", 999999.0, base_time - timedelta(minutes=10))])
time.sleep(6)

spark.sql("SELECT window.start, window.end, region, n, total_amount FROM lab_watermark ORDER BY window.start").show(50, False)
progress = q3.lastProgress or {}
print("eventTime:", json.dumps(progress.get("eventTime", {}), indent=2))
print("stateOperators:", json.dumps(progress.get("stateOperators", []), indent=2))
q3.stop()

## Capstone — transaksi realtime
**Rujukan slide:** Deck 05, slide 40.

Lengkapi sink dan validasi operasional untuk pipeline Kafka → parse → validate → watermark 10 menit → window 5 menit/region.

In [ ]:
valid = parsed.filter(F.col("trx_id").isNotNull() & (F.col("amount") >= 0))
invalid = parsed.filter(F.col("trx_id").isNull() | (F.col("amount") < 0))

capstone = (
    valid.withWatermark("event_time", "10 minutes")
    .groupBy(F.window("event_time", "5 minutes"), "region")
    .agg(F.count("*").alias("transaction_count"), F.sum("amount").alias("total_amount"))
)

capstone.printSchema()
print("TODO: pilih sink, checkpoint persisten, queryName, dan restart test.")
print("TODO: simpan invalid sebagai quarantine beserta metadata Kafka.")
assert len(spark.streams.active) == 0, "Masih ada streaming query yang aktif"
print("Active queries after labs:", len(spark.streams.active))

## Appendix opsional — file stream + checkpoint
Gunakan `file:///home/spark/stream-input` dan checkpoint di `file:///home/spark/outputs` bila Kafka tidak menjadi fokus pengulangan mandiri. Prinsip restart-nya sama: checkpoint menyimpan progress source dan definisi query tidak boleh diubah secara inkompatibel.

## Exit ticket
1. Kapan query streaming benar-benar mulai bekerja?
2. Apa perbedaan event time, Kafka timestamp, dan processing time?
3. Mengapa checkpoint mengalahkan `startingOffsets` saat restart?
4. Apa bukti pada progress bahwa watermark/state berubah?